# 00 — Live Loop Control Panel

**Purpose**: Start, stop, and monitor the live data collection loop from inside the notebook.

All cells run the deterministic engine only.  
`ENABLE_EVOLUTION = False` — no LLM calls, no tool generation, no registry mutations.

**Sections**:
1. Environment check (config, tools, SQLite)
2. Single-shot dry run (one poll, no writes)
3. Start background loop (threaded, interruptible)
4. Stop background loop
5. Status snapshot
6. Tail recent runs from SQLite

## 1. Environment Check

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from config import (
    ENABLE_EVOLUTION,
    SQLITE_ENABLED,
    SQLITE_DB_FILE,
    OUTPUTS_DIR,
    WATCHER_POLL_INTERVAL_SEC,
)
from tools.registry import build_default_registry

registry = build_default_registry()

print(f'ENABLE_EVOLUTION      : {ENABLE_EVOLUTION}')
print(f'SQLITE_ENABLED        : {SQLITE_ENABLED}')
print(f'SQLITE_DB_FILE        : {SQLITE_DB_FILE}')
print(f'OUTPUTS_DIR           : {OUTPUTS_DIR}')
print(f'Default poll interval : {WATCHER_POLL_INTERVAL_SEC}s')
print(f'Registered tools ({len(registry.tool_names)}):')
for name in registry.tool_names:
    tag = ' [generated]' if name in registry.generated_tool_names else ''
    print(f'  * {name}{tag}')

assert not ENABLE_EVOLUTION, 'STOP: Evolution must be OFF for data collection mode.'
print()
print('OK — safe to run live loop.')

## 2. Single-Shot Dry Run

Fetches one round of markets, scores them, prints results.  
**Nothing is written to disk** (`dry_run=True`).

In [ ]:
from prediction_agent.live.run_live_loop import (
    _build_equal_weight_formula,
    _process_market,
    _load_seen_hashes,
)
from api.kalshi_client import KalshiClient

# --- Configuration -----------------------------------------------------------
MARKET_FILTER = None      # e.g. 'KXNBA' to restrict to NBA markets
THRESHOLD     = 0.55
MAX_MARKETS   = 10        # keep small for a quick check
DRY_RUN       = True      # <--- no writes
# -----------------------------------------------------------------------------

client   = KalshiClient()
formula  = _build_equal_weight_formula(registry.tool_names, threshold=THRESHOLD)
seen     = _load_seen_hashes()

try:
    markets = client.get_markets(limit=MAX_MARKETS, status='open')
except Exception as e:
    print(f'API error: {e}')
    markets = []

if MARKET_FILTER:
    markets = [m for m in markets if MARKET_FILTER.lower() in m.get('market_id','').lower()]

print(f'Fetched {len(markets)} markets')
results = []
for mkt in markets:
    r = _process_market(mkt, registry, formula, seen, dry_run=DRY_RUN)
    if r:
        results.append(r)

print(f'Processed {len(results)} new snapshots (dry_run={DRY_RUN})')
print()
for r in results:
    bet_flag = '[BET]' if r.get('bet_triggered') else '     '
    print(f"{bet_flag} {r['market_id']:<30s}  price={r['price']:.3f}  score={r['score']:.4f}")

## 3. Start Background Loop

Runs `run_live_loop()` in a daemon thread so the notebook stays responsive.  
Set `DRY_RUN = True` to collect data without writing, or `False` to write to SQLite + JSONL.

> **Stop**: Run **Section 4** or interrupt the kernel (`Kernel > Interrupt`).

In [ ]:
import threading
import prediction_agent.live.run_live_loop as _loop_mod
from prediction_agent.live.run_live_loop import run_live_loop

# --- Configuration -----------------------------------------------------------
POLL_INTERVAL = 60      # seconds between Kalshi API polls
MARKET_FILTER = None    # e.g. 'KXNBA'
THRESHOLD     = 0.55
MAX_MARKETS   = 50
DRY_RUN       = False   # set True to run without writing to disk
# -----------------------------------------------------------------------------

# Reset shutdown flag so a fresh loop can start
_loop_mod._SHUTDOWN = False

def _loop_target():
    run_live_loop(
        poll_interval=POLL_INTERVAL,
        market_filter=MARKET_FILTER,
        dry_run=DRY_RUN,
        threshold=THRESHOLD,
        max_markets=MAX_MARKETS,
    )

_loop_thread = threading.Thread(target=_loop_target, daemon=True, name='live-loop')
_loop_thread.start()

print(f'Live loop started (thread: {_loop_thread.name}, poll_interval={POLL_INTERVAL}s, dry_run={DRY_RUN})')
print('Run Section 4 to stop gracefully.')

## 4. Stop Background Loop

In [ ]:
import prediction_agent.live.run_live_loop as _loop_mod

_loop_mod._SHUTDOWN = True
print('Shutdown signal sent.')

# Wait up to 5s for the thread to notice
if '_loop_thread' in dir():
    _loop_thread.join(timeout=5)
    status = 'stopped' if not _loop_thread.is_alive() else 'still running (will exit after current iteration)'
    print(f'Loop thread: {status}')
else:
    print('No loop thread found in this session.')

## 5. Status Snapshot

Equivalent to `python -m prediction_agent.live.status` — all data from SQLite.

In [ ]:
from prediction_agent.live.status import _collect_status, print_status

metrics = _collect_status()
print_status(metrics)

## 6. Tail Recent Runs from SQLite

In [ ]:
import pandas as pd
from config import SQLITE_DB_FILE

N_ROWS = 20  # how many recent runs to show

if SQLITE_DB_FILE.exists():
    from prediction_agent.storage.sqlite_store import SQLiteStore
    store = SQLiteStore()

    rows = store.query(
        'SELECT run_id, timestamp, market_id, score, threshold, bet_triggered, outcome '
        'FROM runs ORDER BY timestamp DESC LIMIT ?',
        (N_ROWS,)
    )

    if rows:
        df = pd.DataFrame(rows)
        df['bet_triggered'] = df['bet_triggered'].astype(bool)
        df['score'] = df['score'].round(4)
        print(f'Last {len(df)} runs:')
        print(df.to_string(index=False))

        bets = df['bet_triggered'].sum()
        print(f'\nBets triggered: {bets}/{len(df)} ({100*bets/len(df):.1f}%)')
    else:
        print('No runs in SQLite yet. Start the loop (Section 3) to collect data.')
else:
    print(f'SQLite not found at {SQLITE_DB_FILE}. Start the loop to create it.')